In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
import sys
import asyncio
from IPython.display import Markdown, display
from datetime import datetime
from openai import AsyncOpenAI
from agents import set_default_openai_client

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

load_dotenv(override=True)

groq_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)
set_default_openai_client(groq_client)
print("✅ Using Groq with llama-3.3-70b-versatile")

✅ Using Groq with llama-3.3-70b-versatile


In [2]:
import shutil
npx_path = shutil.which("npx") or shutil.which("npx.cmd")

os.makedirs("./memory", exist_ok=True)

params = {
    "command": npx_path,
    "args": ["-y", "mcp-memory-libsql"],
    "env": {"LIBSQL_URL": "file:./memory/ed.db"}
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', description='Create multiple new entities in the knowledge graph', input_schema={'properties': {'entities': {'items': {'properties': {'name': {'title': 'Name', 'type': 'string'}, 'entityType': {'title': 'EntityType', 'type': 'string'}, 'observations': {'items': {'type': 'string'}, 'title': 'Observations', 'type': 'array'}}, 'required': ['name', 'entityType', 'observations'], 'title': 'Entity', 'type': 'object'}, 'title': 'Entities', 'type': 'array'}}, 'required': ['entities'], 'title': 'create_entitiesArguments', 'type': 'object'}),
 Tool(name='create_relations', description='Create multiple new relations between entities', input_schema={'properties': {'relations': {'items': {'properties': {'from': {'title': 'From', 'type': 'string'}, 'to': {'title': 'To', 'type': 'string'}, 'relationType': {'title': 'RelationType', 'type': 'string'}}, 'required': ['from', 'to', 'relationType'], 'title': 'Relation', 'type': 'object'}, 'title': 'Relations', 'type': 'array'}

In [3]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "groq/llama-3.3-70b-versatile"

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I've stored the information about you in my knowledge graph. I'll remember that you're Ed, an LLM engineer teaching a course about AI Agents, including the MCP protocol. MCP is a protocol for connecting agents with tools, resources and prompt templates, making it easy to integrate AI agents with capabilities.

In [4]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

Based on my knowledge graph, I know that you're Ed, an LLM engineer who teaches a course about AI Agents. I also know that you're familiar with the MCP protocol, which connects agents with tools, resources, and prompt templates to integrate AI capabilities.

In [5]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
if polygon_api_key:
    print("POLYGON_API_KEY is set")
    print("✅ Using free Polygon plan")
else:
    print("❌ POLYGON_API_KEY not set")

POLYGON_API_KEY is set
✅ Using free Polygon plan


In [6]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

Agg(open=175.42, high=177.36, low=174.82, close=176.08, volume=58432900, vwap=176.15, timestamp=1741536000000, transactions=984321, otc=None)

In [7]:
from market import get_share_price
get_share_price("AAPL")

176.08

In [8]:
params = {"command": sys.executable, "args": ["market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

[Tool(name='lookup_share_price', description='This tool provides the current price of the given stock symbol.', input_schema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'})]

In [9]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "groq/llama-3.3-70b-versatile"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The current share price of Apple (AAPL) is $176.08.